# Receptor co-response graph (architecture A) — results

**Compute runs externally**; this notebook only loads the results and plots them.

Earlier graphs were built in EMBEDDING space (x) — the boost's own territory → null. Here the graph
comes from **y**: two receptors are neighbours if activated by **similar molecule sets** (cosine over
their signed columns of the train interaction matrix). This *functional / pharmacological* similarity
need not match ESM sequence similarity — and it is the one y-structure a **cold** molecule can reach,
since receptors stay warm in inductive.

The run isolates one question — **does smoothing a molecule's binding profile over CO-RESPONSE (y)
receptor neighbours beat smoothing over ESM (x) receptor neighbours?** Same mechanism, only the
neighbour source differs. Bars:

- **raw** — `[mol | prot]`.
- **nbr** — + molecule-neighbour CF read at the target protein (does a profile help at all).
- **nbr_esm** — nbr + profile smoothed over **ESM** receptor neighbours (x-space control).
- **nbr_cor** — nbr + profile smoothed over **co-response** receptor neighbours (**architecture A**).
- **own_esm / own_cor** *(transductive only)* — the query's OWN binding row smoothed over ESM vs
  co-response receptors (purest functional-vs-sequence test; needs a warm query).

Both regimes, **5 repeats** each. Inductive cold split is **stratified to matched prevalence**.
Aggregation weights are **binary**. Embeddings standardised.

### Run the compute
```bash
bash scripts/modeling/eval/run_receptor_coresponse_graph.sh            # live dashboard
MAX_PARALLEL=10 bash scripts/modeling/eval/run_receptor_coresponse_graph.sh
```
Then run the cells below.

In [ ]:
import sys, pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results/graph/receptor_coresponse'
COMBINED = RESULT_DIR / 'tables/receptor_coresponse_runs.csv'
RUNS_DIR = RESULT_DIR / 'tables/runs'
FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

# prefer the combined CSV; fall back to globbing per-unit CSVs (partial runs still render)
if COMBINED.exists():
    RES = pd.read_csv(COMBINED)
    print('loaded', COMBINED.relative_to(ROOT))
elif RUNS_DIR.exists() and glob.glob(str(RUNS_DIR / '*.csv')):
    RES = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RUNS_DIR / '*.csv')))], ignore_index=True)
    print(f'loaded {RES.repeat.nunique()} repeat-files from', RUNS_DIR.relative_to(ROOT))
else:
    RES = pd.DataFrame()
    print('No results yet. Run:  bash scripts/modeling/eval/run_receptor_coresponse_graph.sh')

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

if len(RES):
    n = RES.groupby('regime').repeat.nunique().to_dict()
    print('repeats per regime:', n)
    print('mean test prevalence:', RES.groupby('regime').test_prevalence.mean().round(3).to_dict())
    # how different is the functional (y) receptor graph from the sequence (ESM) one?
    diag = RES.groupby('regime')[['coresp_mean_sim', 'esm_coresp_jaccard']].mean().round(3)
    print('\nreceptor-graph diagnostics (co-response mean-sim; ESM/co-response neighbour Jaccard):')
    display(diag)
    display(RES.groupby(['regime', 'bar'])[METRICS].mean().round(3))

In [ ]:
# ---- bar-grid: regimes (rows) x metrics (cols), mean +/- 95% t-CI over repeats ----
ORDER = ['raw', 'nbr', 'nbr_esm', 'nbr_cor', 'own_esm', 'own_cor']
LABEL = {'raw': 'raw', 'nbr': 'nbr', 'nbr_esm': 'nbr\n+ESM', 'nbr_cor': 'nbr\n+co-resp',
         'own_esm': 'own\n+ESM', 'own_cor': 'own\n+co-resp'}
# grey baseline; ESM (sequence, x) in blue; co-response (functional, y) highlighted in orange
BAR_C = {'raw': '#9a9a95', 'nbr': '#c8c8c2', 'nbr_esm': '#2a78d6', 'nbr_cor': '#e07b39',
         'own_esm': '#7fa8dd', 'own_cor': '#f0a868'}

if not len(RES):
    print('No results to plot yet.')
else:
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(15, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        bars = [b for b in ORDER if b in set(d.bar)]          # own-bars only present in transductive
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in bars]
            errs = [ci95_half(d[d.bar == b][metric]) for b in bars]
            x = np.arange(len(bars))
            ax.bar(x, means, color=[BAR_C[b] for b in bars], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            if means:
                ax.axhline(means[0], color=BAR_C['raw'], ls='--', lw=1, zorder=1)   # raw baseline
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([LABEL[b] for b in bars], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} repeats)', fontsize=10)
    fig.suptitle('Receptor co-response graph (y) vs ESM graph (x) — mean +/- 95% t-CI '
                 '(stratified inductive split; orange = functional/co-response)', fontsize=12, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'receptor_coresponse_graph.png', dpi=150, bbox_inches='tight')
    plt.show()

## How to read

- The key contrast is **nbr_cor (orange, y-graph) vs nbr_esm (blue, x-graph)** in **inductive_molecule** —
  the only regime where the receptor graph is the cold molecule's sole y-lever. If co-response > ESM, the
  functional receptor neighbourhood carries signal the sequence embedding misses.
- **own_cor vs own_esm** (transductive) is the purest test — it smooths the query's *real* binding row over
  each receptor graph. It needs a warm query, so it is absent in inductive.
- The **diagnostics** cell reports how much the two receptor graphs even differ (`esm_coresp_jaccard` low ⇒
  functional ≠ sequence, so the contrast is meaningful; near 1 ⇒ the two graphs coincide and no difference
  is expected).
- Compare on **AUROC** primarily (prevalence-independent; the stratified split matches val/test prevalence).
  Whiskers are 95% t-CI over 5 repeats. Given the transductive ceiling (~0.89) and the inductive cold-start
  wall, expect small deltas — the scientific read (functional vs sequence) matters even if predictive gain is modest.

Knobs via env: `K` (molecule neighbours), `KP` (receptor neighbours), `MAX_PARALLEL`. Delete
`results/graph/receptor_coresponse/tables/` to regenerate.